In [11]:
from common_utils.utils import chunks
from tqdm import tqdm
from matplotlib import pyplot as plt
from pylab import rcParams
import seaborn as sns

from IPython.display import display, HTML
import pandas as pd

import logging
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import pickle

from common_utils.notebook_utils import sign_notebook

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 200)
from pandas.io.json import json_normalize


from dbutils.query_utils import get_interval_clauses, run_select
from dbutils.utils import get_nabu_payments_client, get_nabu_ledger_client, get_data_science_bigquery_client, get_model_bucket_client
from dbutils.query_utils import get_select_query, run_select, get_interval_clauses
from customers_analytics.fraud_analysis.fraud_db_utils import get_fraud_client
from customers_analytics.nabu.nabu_db_utils import (
    get_nabu_client_internal, 
    get_nabu_backoffice_client, 
    get_nabu_client, 
    get_nabu_client_private, 
    get_nabu_backoffice_client_prod, 
    categorize_blockchain_error_reasons
)

from db_schemas.schema_utils import get_metadata_from_source_name, get_engine_from_source_name, SourceType
source_name, source_type = "nabu-payments", SourceType.POSTGRES


In [5]:
PATH_DIR = "./pkl/"

In [6]:
def filename_from_source_name(source_name):
    return f"{source_name.replace('-', '_')}_meta.pkl"

def update_metadata_pickle(source_name, schema=None):
    logging.info(f"Creating/Updating pickle for {source_name} and schema {schema}")
    try:
        metadata = get_metadata_from_source_name(source_name, schema=schema, use_pickle=False)
    except SourceMissingException:
        logging.error(f"Could not update source {source_name}, probably due to permissions")
        return

    filename = filename_from_source_name(source_name)
    destination = f"{PATH_DIR}/{filename}"

    with open(destination, "wb") as f:
        pickle.dump(metadata, f)

    logging.info(f"Successfully created/updated pickles for {source_name} and schema {schema}")

## Create pickle with `sqlalchemy.sql.schema.MetaData` object based on provided schema

In [8]:
source_name, source_type = "nabu-payments", SourceType.POSTGRES
engine = get_engine_from_source_name(source_name, source_type)

In [9]:
metadata = get_metadata_from_source_name(source_name, schema="payments", use_pickle=True)

metadata.reflect(bind=engine, views=True)

metadata.tables["payments.account"]

Table('account', MetaData(bind=None), Column('id', UUID(), table=<account>, primary_key=True, nullable=False), Column('type', TEXT(), table=<account>, nullable=False), Column('partner', TEXT(), table=<account>, nullable=False), Column('product', TEXT(), table=<account>, nullable=False), Column('account_ref', TEXT(), table=<account>, nullable=False), Column('currency', TEXT(), table=<account>, nullable=False), Column('agent_ref', TEXT(), table=<account>), Column('extra_attributes', JSONB(astext_type=Text()), table=<account>), Column('user', UUID(), table=<account>), Column('inserted_at', TIMESTAMP(), table=<account>, nullable=False), Column('updated_at', TIMESTAMP(), table=<account>, nullable=False), Column('last_state', TEXT(), table=<account>), schema='payments')

In [12]:
update_metadata_pickle(source_name=source_name, schema="payments")

INFO:root:Creating/Updating pickle for nabu-payments and schema payments
INFO:root:Successfully created/updated pickles for nabu-payments and schema payments


## Get `GoogleBucketClient` client and create a connection to BucketSource Storage

In [2]:
bucket = get_model_bucket_client()

## Get Last N blob's elements from list 

In [41]:
bucket.list_files()[-5:]

['ModelFit_feaafe8b1f3c64f3852cc17490afe2b3ca413bfd1df0ff62e5308d3e1e734769.pkl',
 'ModelFit_feaf9011ce2e2501e15138e2f1d7094c41b0b0f4d6ce8c8367eb4674a811182f.pkl',
 'ModelFit_fecfe2092d98a9c0988b977920df4f2ceee79e2afc57a745ca2b2940c9d82e75.pkl',
 'ModelFit_ff5f1d49614acb3023a67e1eaf13232260263ccd444f5f8322c5fd3a39ac2318.pkl',
 'nabu_payments_meta.pkl']

In [40]:
[blob for blob in bucket._bucket.list_blobs()][-4:]

[<Blob: blockchain-data-science-models, ModelFit_feaf9011ce2e2501e15138e2f1d7094c41b0b0f4d6ce8c8367eb4674a811182f.pkl, 1576108974885170>,
 <Blob: blockchain-data-science-models, ModelFit_fecfe2092d98a9c0988b977920df4f2ceee79e2afc57a745ca2b2940c9d82e75.pkl, 1590883719473103>,
 <Blob: blockchain-data-science-models, ModelFit_ff5f1d49614acb3023a67e1eaf13232260263ccd444f5f8322c5fd3a39ac2318.pkl, 1590278894139129>,
 <Blob: blockchain-data-science-models, nabu_payments_meta.pkl, 1653515958301364>]

## Upload: Add pickle to bucket storage

In [51]:
pkl_file_to_upload = './pkl/nabu_payments_meta.pkl'
destination_name = "nabu_payments_meta.pkl"

bucket.upload_from_file_name(file_name=pkl_file_to_upload)

In [26]:
bucket._bucket.blob("nabu_payments_meta.pkl")

<Blob: blockchain-data-science-models, nabu_payments_meta.pkl, None>

In [50]:
ts = bucket.get_blobs_info()["nabu_payments_meta.pkl"]['time_created']

f"Bucket  created at: {datetime.strftime(ts, '%Y-%m-%d:%H:%M:%S')}"

'Bucket created at: 2022-05-25:21:59:18'

## Download pkl from gbucket

In [56]:
io = bucket.download_to_file_obj(destination_name)

In [57]:
metadata = pickle.load(io)

In [60]:
metadata.tables["payments.account"]

Table('account', MetaData(bind=None), Column('id', UUID(), table=<account>, primary_key=True, nullable=False), Column('type', TEXT(), table=<account>, nullable=False), Column('partner', TEXT(), table=<account>, nullable=False), Column('product', TEXT(), table=<account>, nullable=False), Column('account_ref', TEXT(), table=<account>, nullable=False), Column('currency', TEXT(), table=<account>, nullable=False), Column('agent_ref', TEXT(), table=<account>), Column('extra_attributes', JSONB(astext_type=Text()), table=<account>), Column('user', UUID(), table=<account>), Column('inserted_at', TIMESTAMP(), table=<account>, nullable=False), Column('updated_at', TIMESTAMP(), table=<account>, nullable=False), Column('last_state', TEXT(), table=<account>), schema='payments')